In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date, timedelta
from config import *
from SIR_EAKF_model import *
from seasonal_drift_model import *

In [ ]:
# season_epiweeks = iso_week_tuples(2023, 41, 2024, 15) #2024 season
# season_epiweeks = iso_week_tuples(2024, 47, 2025, 22) #2025 season
season_epiweeks = iso_week_tuples(2025, 47, 2026, 21) #2026 season

weeks = [w[1] for w in season_epiweeks]
weeks_to_predict = len(weeks[weeks.index(epiweek):])

In [ ]:
ref_date = epiweek_to_dates(epiyear, epiweek).enddate() #Saturday at the end of epiweek
ref_date = pd.Timestamp(ref_date)

new_format = True
download_hosp=False
fix_partial_reporting=False
fix_outliers=False
df_hosp = read_hosp_incidence_data(data_dir, epiyear, epiweek, states, 
                                new_format=new_format, download=download_hosp,
                                fix_partial_reporting=fix_partial_reporting, 
                                fix_outliers=fix_outliers,
                                plot=False)
df_ili = read_ili_incidence_data(data_dir, epiyear, epiweek, states, df_hosp, 
                            smooth=False, scale=True, regress=True, plot=False)

df_hosp = df_hosp[df_hosp.date<pd.to_datetime(ref_date)]
df_ili = df_ili[df_ili.date<pd.to_datetime(ref_date)]

switch_epiyear = 2022
switch_epiweek = 26
switch_date = pd.to_datetime(epiweek_to_dates(switch_epiyear, switch_epiweek).enddate())

df1 = df_ili[df_ili.date < switch_date]
df2 = df_hosp[(df_hosp.date >= switch_date)]
df_hosp_ex = pd.concat([df1, df2])

hosp_ex_start_date = pd.to_datetime('2010-10-09', format="%Y-%m-%d")
df_hosp_ex = df_hosp_ex[df_hosp_ex['date']>=hosp_ex_start_date]
df_hosp_ex = df_hosp_ex.reset_index(drop=True)
df_hosp_ex[states] = df_hosp_ex[states].fillna(0)

covid_start_epiyear = 2020
covid_start_epiweek = 26
covid_start_date = pd.to_datetime(epiweek_to_dates(covid_start_epiyear, covid_start_epiweek).enddate())
covid_end_date = switch_date
df_hosp_ex = df_hosp_ex[~df_hosp_ex["date"].between(covid_start_date, covid_end_date, inclusive="left")]

dat_changerate_ref = df_hosp_ex[df_hosp_ex.date==ref_date-timedelta(weeks=1)]

df_hosp_ex_long = pd.melt(df_hosp_ex,id_vars=['date','year','week'],value_vars=df_hosp_ex.columns[3:],var_name='location',value_name='value')
df_hosp_ex_long['location'] = df_hosp_ex_long['location'].map(abbr2loc)
keys = pd.DataFrame(season_epiweeks, columns=["year","week"]).drop_duplicates()
df_hosp_ex_long = df_hosp_ex_long.merge(keys, on=["year","week"], how="inner")
# df_hosp_ex_long

In [ ]:
#generate pred using seasonal drift model
seasonal_drift_model = "seasonal_drift"
pool_weight=0.5
epiweek_window = 1
df_pred, df_pred_samples = generate_seasonal_drift_pred(
                df_hosp_ex, ref_date, weeks_to_predict, locations, quantiles, num_samples, epiweek_window,
                dat_changerate_ref, results_dir, model_desc=seasonal_drift_model, 
                pool_weight=pool_weight, generate_qual_pred=False, save_results=False,
                return_samples=True,  random_state=100)

sir_eakf_model = "SIR-EAKF"
sir_eakf_start_date = pd.to_datetime('2025-10-25', format="%Y-%m-%d")  #pd.to_datetime('2024-09-01', format="%Y-%m-%d") #
df_pred2, df_pred_samples2 = generate_sir_eakf_pred(df_hosp_ex, sir_eakf_start_date, ref_date, weeks_to_predict, 
                        locations, quantiles, num_samples, 
                        dat_changerate_ref, results_dir, model_desc=sir_eakf_model,
                        generate_qual_pred=False, save_results=False,
                        return_samples=True,  random_state=100)

In [ ]:
df_pred_us = df_pred[df_pred.location=='US']
df_pred_us_mean = df_pred_us[df_pred_us.output_type_id==0.5]
plt.figure(figsize=(12,6))
plt.plot(df_pred_us_mean.target_end_date,df_pred_us_mean.value)
x = plt.xticks(rotation=45, ha="right")

df_pred2_us = df_pred2[df_pred2.location=='US']
df_pred2_mean = df_pred2_us[df_pred2_us.output_type_id==0.5]
plt.figure(figsize=(12,6))
plt.plot(df_pred2_mean.target_end_date,df_pred2_mean.value)
x = plt.xticks(rotation=45, ha="right")

In [ ]:
def plot_curves(df_samples_long, location, sample_ids=None, use_dates=True):
    loc = str(location).zfill(2)
    work = df_samples_long[df_samples_long["location"] == loc].copy()

    if sample_ids is not None:
        sid_set = set(sample_ids)
        work = work[work["sample_id"].isin(sid_set)]

    # ensure proper dtypes & uniqueness
    work["horizon"] = pd.to_numeric(work["horizon"], errors="coerce")
    work["target_end_date"] = pd.to_datetime(work["target_end_date"], errors="coerce")
    work = (work
            .sort_values(["sample_id", "horizon"])
            .drop_duplicates(["sample_id", "horizon"], keep="last"))

    xcol = "target_end_date" if use_dates else "horizon"

    fig, ax = plt.subplots()
    for _, g in work.groupby("sample_id", sort=False):
        ax.plot(g[xcol].to_numpy(), g["value"].to_numpy(), linewidth=0.8, alpha=0.3)

    ax.set_xlabel(xcol)
    ax.set_ylabel("value")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_curves(df_pred_samples, 'US', None, use_dates=False)
plot_curves(df_pred_samples, 'US', [0,1,2,3,4,5,6,7,8,9], use_dates=False)

plot_curves(df_pred_samples2, 'US', None, use_dates=False)
plot_curves(df_pred_samples2, 'US', [0,1,2,3,4,5,6,7,8,9], use_dates=False)

In [ ]:
def augment_samples_with_nb(
    df_samples_long: pd.DataFrame,
    R: int,
    phi: float,
    *,
    min_mu: float = 1e-6,
    random_state: int | None = None,
    value_col_in: str = "value",
    value_col_out: str = "value"
) -> pd.DataFrame:
    """
    Expand long samples with R observation replicates per sample using a
    Negative-Binomial observation model (Gamma–Poisson mixture).

    Parameters
    ----------
    df_samples_long : DataFrame
        Columns: ['reference_date','location','horizon','target_end_date','sample_id', value_col_in]
    R : int
        Number of observation replicates per (horizon, sample_id).
    phi : float
        NB dispersion (size) parameter. Var = mu + mu^2/phi. Larger phi => milder overdispersion.
    min_mu : float, default 1e-6
        Floor for tiny means to avoid degenerate draws when mu ~ 0.
    random_state : int | None
        Seed for reproducibility.
    value_col_in : str
        Column name of latent mean (usually 'value').
    value_col_out : str
        Column name for the observed replicate (usually 'value' to overwrite).

    Returns
    -------
    DataFrame
        Same columns as input, plus 'replicate_id'. The 'sample_id' is made unique:
        new_sample_id = old_sample_id * R + replicate_id.
        The `value_col_out` contains the **observed** counts drawn from NB.
    """
    if R <= 0:
        raise ValueError("R must be a positive integer.")
    if phi <= 0:
        raise ValueError("phi must be positive.")

    df = df_samples_long.copy()

    # Dtypes / normalization
    df["reference_date"]  = pd.to_datetime(df["reference_date"])
    df["target_end_date"] = pd.to_datetime(df["target_end_date"])
    df["horizon"]         = pd.to_numeric(df["horizon"], errors="coerce").astype(int)
    df["location"]        = df["location"].astype(str).str.zfill(2)
    if value_col_in not in df.columns:
        raise KeyError(f"{value_col_in!r} not found in df_samples_long columns")

    # Latent mean (counts), floored for stability
    mu = df[value_col_in].to_numpy(dtype=float)
    mu = np.maximum(mu, float(min_mu))

    rng = np.random.default_rng(random_state)
    out_frames = []

    # Gamma–Poisson: lambda ~ Gamma(shape=phi, scale=mu/phi); Y ~ Poisson(lambda)
    # Vectorized per replicate
    shape = float(phi)
    scale = mu / float(phi)

    for r in range(R):
        lam = rng.gamma(shape=shape, scale=scale, size=mu.size)
        y   = rng.poisson(lam, size=mu.size)

        df_r = df.copy()
        df_r["replicate_id"] = r
        # make sample_id unique across replicates (keep original if you also want to track it)
        if "original_sample_id" not in df_r.columns:
            df_r["original_sample_id"] = df_r["sample_id"]
        df_r["sample_id"] = df_r["original_sample_id"].astype(int) * R + r
        df_r[value_col_out] = y.astype(float)  # keep float dtype for consistency

        out_frames.append(df_r)

    df_aug = pd.concat(out_frames, ignore_index=True)

    # Optional: ensure column order feels natural
    cols = ["reference_date","location","horizon","target_end_date",
            "sample_id","replicate_id","original_sample_id"]
    # move known columns to the front if present
    front = [c for c in cols if c in df_aug.columns]
    others = [c for c in df_aug.columns if c not in front]
    df_aug = df_aug[front + others]

    return df_aug


In [ ]:
def season_peak_week_pmf_and_peak_quantiles(
    df_samples_long: pd.DataFrame,
    observed_long: pd.DataFrame,
    quantiles
) -> pd.DataFrame:
    """
    Build (1) season-long peak-week PMF and (2) peak-incidence quantile forecasts
    for each (reference_date, location), using the same winner logic.

    Inputs
    ------
    df_samples_long : DataFrame (Monte Carlo samples at each reference_date)
        Columns: ['reference_date','location','horizon','target_end_date','sample_id','value']
    observed_long : DataFrame (observed history for the season)
        Columns: ['date','year','week','location','value']  (or 'target_end_date' instead of 'date')

    Returns
    -------
    DataFrame with rows for both targets, unified schema:
      ['reference_date','target','horizon','target_end_date',
       'location','output_type','output_type_id','value']

    Conventions
    -----------
    - PMF:
        target            = 'peak week inc flu hosp'
        output_type       = 'pmf'
        output_type_id    = date (datetime64[ns]) of the peak week bin
        horizon, target_end_date = '' (empty strings)
        Includes ALL weeks from earliest observed to last sample week (zeros padded).
        Sums to exactly 1 per (reference_date, location).
    - Peak incidence quantiles:
        target            = 'peak inc flu hosp'
        output_type       = 'quantile'
        output_type_id    = quantile level (float)
        horizon, target_end_date = '' (empty strings)
        Computed from the per-sample winner peak values.
    """

    # ---------- normalize dtypes ----------
    s = df_samples_long.copy()
    s["reference_date"]  = pd.to_datetime(s["reference_date"])
    s["target_end_date"] = pd.to_datetime(s["target_end_date"])
    s["horizon"]         = pd.to_numeric(s["horizon"], errors="coerce").astype(int)
    s["location"]        = s["location"].astype(str).str.zfill(2)

    obs = observed_long.copy()
    if "date" not in obs.columns and "target_end_date" in obs.columns:
        obs = obs.rename(columns={"target_end_date": "date"})
    obs["date"]     = pd.to_datetime(obs["date"])
    obs["location"] = obs["location"].astype(str).str.zfill(2)

    out_rows = []
    s = s.sort_values(["reference_date","location","sample_id","horizon"])

    # infer weekly anchor from samples (e.g., 'W-SAT')
    def _infer_weekly_freq(dates: pd.Series) -> str:
        if dates.empty:
            return "7D"
        dow = dates.dt.dayofweek.mode().iloc[0]  # 0=Mon,...,6=Sun
        abbr = ["MON","TUE","WED","THU","FRI","SAT","SUN"][dow]
        return f"W-{abbr}"

    for (ref_date, loc), g in s.groupby(["reference_date","location"], sort=False):
        # Observed-to-date for this location
        obs_g = obs[(obs["location"] == loc) & (obs["date"] < ref_date)].copy()

        # Season bounds from earliest observed (for this loc) to last sample date (for this ref/loc)
        sample_dates = g["target_end_date"].dropna().sort_values()
        if sample_dates.empty:
            continue
        season_end = sample_dates.max()
        season_start = obs_g["date"].min() if not obs_g.empty else sample_dates.min()

        freq = _infer_weekly_freq(sample_dates)
        full_dates = pd.date_range(season_start, season_end, freq=freq)

        # ------- winners per sample (date + value) -------
        if obs_g.empty:
            # No observed history: winner is future peak in each path
            peak_idx = g.groupby("sample_id")["value"].idxmax()
            peaks = g.loc[peak_idx, ["sample_id", "value", "target_end_date"]].copy()
            peaks.rename(columns={"value": "winner_peak_value",
                                  "target_end_date": "winner_peak_date"}, inplace=True)
        else:
            # Observed current peak (first max: earliest date on ties)
            obs_sorted = obs_g.sort_values(["value", "date"], ascending=[False, True])
            y_star = float(obs_sorted.iloc[0]["value"])
            w_star_date = pd.to_datetime(obs_sorted.iloc[0]["date"])

            # Future peaks per sample (first maximum in simulated future)
            peak_idx = g.groupby("sample_id")["value"].idxmax()
            fut = g.loc[peak_idx, ["sample_id", "value", "target_end_date"]].copy()
            fut.rename(columns={"value": "future_max",
                                "target_end_date": "future_peak_date"}, inplace=True)

            # Dethrone if strict future > observed (ties keep observed if tie_keeps_observed=True)
            tie_keeps_observed = True
            dethroned = (fut["future_max"] > y_star) if tie_keeps_observed else (fut["future_max"] >= y_star)

            # Build winners with proper dtypes
            winners = fut[["sample_id", "future_max", "future_peak_date"]].copy()
            winners["winner_peak_date"]  = winners["future_peak_date"]
            winners["winner_peak_value"] = winners["future_max"]
            # where not dethroned → keep observed peak (date & value)
            winners.loc[~dethroned.values, "winner_peak_date"]  = pd.Timestamp(w_star_date)
            winners.loc[~dethroned.values, "winner_peak_value"] = y_star

            peaks = winners[["sample_id", "winner_peak_date", "winner_peak_value"]].copy()

        # If no observed history branch was used, 'peaks' was set already.
        if obs_g.empty:
            peaks["winner_peak_date"]  = peaks["winner_peak_date"]  # rename clarity
            peaks["winner_peak_value"] = peaks["winner_peak_value"] if "winner_peak_value" in peaks.columns else peaks["value"]

        # ------- PMF over peak dates (pad to full season grid) -------
        pmf_counts = (peaks["winner_peak_date"]
                      .value_counts()
                      .rename_axis("peak_date")
                      .to_frame("count")
                      .reset_index())
        n_samples = float(g["sample_id"].nunique())
        pmf_counts["value"] = pmf_counts["count"] / n_samples
        pmf_counts = pmf_counts[["peak_date","value"]]

        full = pd.DataFrame({"peak_date": full_dates})
        probs_full = full.merge(pmf_counts, on="peak_date", how="left").fillna({"value": 0.0})

        # exact renormalization for PMF
        total = probs_full["value"].sum()
        if total > 0:
            probs_full["value"] = probs_full["value"] / total

        # format PMF rows
        pmf_rows = probs_full.copy()
        pmf_rows.insert(0, "reference_date", ref_date)
        pmf_rows.insert(1, "target", "peak week inc flu hosp")
        pmf_rows["horizon"] = ""
        pmf_rows["target_end_date"] = ""
        pmf_rows["location"] = loc
        pmf_rows["output_type"] = "pmf"
        pmf_rows["output_type_id"] = pmf_rows["peak_date"]  # keep as datetime; you can .dt.strftime if you prefer
        pmf_rows = pmf_rows.drop(columns=["peak_date"])

        out_rows.append(pmf_rows)

        # ------- Peak incidence quantiles (from winner_peak_value) -------
        # Collect winner peak values for this (ref_date, loc)
        if "winner_peak_value" not in peaks.columns:
            # (obs_g.empty branch) rename/ensure
            if "value" in peaks.columns and "winner_peak_value" not in peaks.columns:
                peaks = peaks.rename(columns={"value": "winner_peak_value"})
        vals = peaks["winner_peak_value"].to_numpy(dtype=float)

        if vals.size > 0:
            qvals = np.quantile(vals, quantiles)
            qdf = pd.DataFrame({
                "reference_date": ref_date,
                "target": "peak inc flu hosp",
                "horizon": "",
                "target_end_date": "",
                "location": loc,
                "output_type": "quantile",
                "output_type_id": np.array(quantiles, dtype=float),
                "value": qvals.astype(float)
            })
            out_rows.append(qdf)

    # unify
    out = pd.concat(out_rows, ignore_index=True)

    # split by target to avoid mixed dtype issues in output_type_id
    pmf = out[out["target"] == "peak week inc flu hosp"].copy()
    qnt = out[out["target"] == "peak inc flu hosp"].copy()

    # --- PMF: make dates into strings for consistent ordering ---
    # (or keep as datetime and sort with a separate key; strings are simplest)
    pmf["output_type_id"] = pd.to_datetime(pmf["output_type_id"]).dt.strftime("%Y-%m-%d")
    pmf = pmf.sort_values(
        ["reference_date", "location", "target", "output_type", "output_type_id"]
    )

    # --- Quantiles: ensure numeric order on quantile levels ---
    qnt["output_type_id"] = pd.to_numeric(qnt["output_type_id"], errors="coerce")
    qnt = qnt.sort_values(
        ["reference_date", "location", "target", "output_type", "output_type_id"]
    )

    # stitch back
    out = pd.concat([pmf, qnt], ignore_index=True)

    # for PMF rows only: enforce exact sums == 1 per (reference_date, location)
    is_pmf = out["target"].eq("peak week inc flu hosp")
    if is_pmf.any():
        sums = out.loc[is_pmf].groupby(["reference_date","location"])["value"].transform("sum")
        out.loc[is_pmf, "value"] = np.where(sums > 0, out.loc[is_pmf, "value"] / sums, out.loc[is_pmf, "value"])

    # final column order (unchanged)
    out = out[[
        "reference_date","target","horizon","target_end_date",
        "location","output_type","output_type_id","value"
    ]]
    return out

In [ ]:
def season_peak_week_pmf_and_peak_quantiles(
    df_samples_long: pd.DataFrame,
    observed_long: pd.DataFrame,
    quantiles,
    augment: bool = False,
) -> pd.DataFrame:
    """
    Returns BOTH:
      (1) season-long peak-week PMF  -> target='peak week inc flu hosp', output_type='pmf'
      (2) peak-incidence quantiles   -> target='peak inc flu hosp',     output_type='quantile'

    Important:
      - If augment is True, and augmented samples dataframe will be generated and used 
        ONLY for the PMF (peak WEEK) selection.
        Peak INCIDENCE quantiles are ALWAYS computed using df_samples_long (latent),
        which avoids winner's-curse inflation.
      - Past weeks are padded with zero probability; PMF covers all weeks from earliest
        observed to the last sample week (determined from the PMF dataset).

    Inputs
    ------
    df_samples_long : DataFrame of latent samples at each reference_date.
        Columns: ['reference_date','location','horizon','target_end_date','sample_id','value']
    observed_long   : DataFrame of observed history for the season.
        Columns: ['date','year','week','location','value'] (or 'target_end_date' instead of 'date')
    quantiles       : iterable of quantile levels (floats in (0,1))
    df_samples_long_aug : (optional) DataFrame with the same schema as df_samples_long.
        If provided, used for PMF computation only.

    Returns
    -------
    DataFrame with unified schema:
      ['reference_date','target','horizon','target_end_date',
       'location','output_type','output_type_id','value']
    Conventions
    -----------
    PMF rows:
      - target='peak week inc flu hosp'
      - output_type='pmf'
      - output_type_id = ISO date string 'YYYY-MM-DD' of the peak week bin
      - horizon='' , target_end_date=''
      - sums to exactly 1 per (reference_date, location)
    Quantile rows:
      - target='peak inc flu hosp'
      - output_type='quantile'
      - output_type_id = quantile level (float)
      - horizon='' , target_end_date=''
    """

    # ---------- normalize observed ----------
    obs = observed_long.copy()
    if "date" not in obs.columns and "target_end_date" in obs.columns:
        obs = obs.rename(columns={"target_end_date": "date"})
    obs["date"]     = pd.to_datetime(obs["date"])
    obs["location"] = obs["location"].astype(str).str.zfill(2)

    # ---------- normalize latent samples (used for quantiles) ----------
    s_lat = df_samples_long.copy()
    s_lat["reference_date"]  = pd.to_datetime(s_lat["reference_date"])
    s_lat["target_end_date"] = pd.to_datetime(s_lat["target_end_date"])
    s_lat["horizon"]         = pd.to_numeric(s_lat["horizon"], errors="coerce").astype(int)
    s_lat["location"]        = s_lat["location"].astype(str).str.zfill(2)
    s_lat = s_lat.sort_values(["reference_date","location","sample_id","horizon"])

    # ---------- PMF samples source: use augmented if augment is True, else latent ----------
    if augment:
        s_pmf = augment_samples_with_nb(s_lat,R=10,phi=50)
        s_pmf["reference_date"]  = pd.to_datetime(s_pmf["reference_date"])
        s_pmf["target_end_date"] = pd.to_datetime(s_pmf["target_end_date"])
        s_pmf["horizon"]         = pd.to_numeric(s_pmf["horizon"], errors="coerce").astype(int)
        s_pmf["location"]        = s_pmf["location"].astype(str).str.zfill(2)
        s_pmf = s_pmf.sort_values(["reference_date","location","sample_id","horizon"])
    else:
        s_pmf = s_lat

    out_rows = []

    # infer weekly anchor from the PMF sample dates (e.g., 'W-SAT')
    def _infer_weekly_freq(dates: pd.Series) -> str:
        if dates.empty:
            return "7D"
        dow = dates.dt.dayofweek.mode().iloc[0]  # 0=Mon,...,6=Sun
        abbr = ["MON","TUE","WED","THU","FRI","SAT","SUN"][dow]
        return f"W-{abbr}"

    # --------- iterate by (reference_date, location) ---------
    # PMF is built from s_pmf; quantiles from s_lat (latent)
    for (ref_date, loc), g_pmf in s_pmf.groupby(["reference_date","location"], sort=False):
        # corresponding latent group for quantiles (same ref/location)
        g_lat = s_lat[(s_lat["reference_date"] == ref_date) & (s_lat["location"] == loc)]
        if g_lat.empty or g_pmf.empty:
            continue

        # Observed-to-date for this location/ref
        obs_g = obs[(obs["location"] == loc) & (obs["date"] < ref_date)].copy()

        # Season bounds for PMF padding
        sample_dates = g_pmf["target_end_date"].dropna().sort_values()
        if sample_dates.empty:
            continue
        season_end = sample_dates.max()
        season_start = obs_g["date"].min() if not obs_g.empty else sample_dates.min()

        freq = _infer_weekly_freq(sample_dates)
        full_dates = pd.date_range(season_start, season_end, freq=freq)

        # --------------------- PMF winners (using s_pmf only) ---------------------
        if obs_g.empty:
            # no observed history: winner is future peak in each PMF path
            peak_idx = g_pmf.groupby("sample_id")["value"].idxmax()
            pmf_peaks = g_pmf.loc[peak_idx, ["sample_id", "value", "target_end_date"]].copy()
            pmf_peaks.rename(columns={"value": "future_max",
                                      "target_end_date": "future_peak_date"}, inplace=True)
            pmf_winner_dates = pmf_peaks["future_peak_date"]
        else:
            # observed current peak (first max → earliest date on ties)
            obs_sorted = obs_g.sort_values(["value", "date"], ascending=[False, True])
            y_star = float(obs_sorted.iloc[0]["value"])
            # future maxima per PMF path
            peak_idx = g_pmf.groupby("sample_id")["value"].idxmax()
            fut = g_pmf.loc[peak_idx, ["sample_id", "value", "target_end_date"]].copy()
            fut.rename(columns={"value": "future_max",
                                "target_end_date": "future_peak_date"}, inplace=True)
            # strict dethrone (tie keeps observed)
            dethroned = (fut["future_max"] > y_star)
            pmf_winner_dates = fut["future_peak_date"].where(dethroned, pd.Timestamp(obs_sorted.iloc[0]["date"]))

        # PMF counts over dates
        pmf_counts = (pmf_winner_dates.value_counts()
                      .rename_axis("peak_date")
                      .to_frame("count")
                      .reset_index())
        n_pmf_samples = float(g_pmf["sample_id"].nunique())
        pmf_counts["value"] = pmf_counts["count"] / n_pmf_samples
        pmf_counts = pmf_counts[["peak_date","value"]]

        # pad to full season grid
        full = pd.DataFrame({"peak_date": full_dates})
        probs_full = full.merge(pmf_counts, on="peak_date", how="left").fillna({"value": 0.0})
        # renormalize PMF
        total = probs_full["value"].sum()
        if total > 0:
            probs_full["value"] = probs_full["value"] / total

        # format PMF rows
        pmf_rows = probs_full.copy()
        pmf_rows.insert(0, "reference_date", ref_date)
        pmf_rows.insert(1, "target", "peak week inc flu hosp")
        pmf_rows["horizon"] = ""
        pmf_rows["target_end_date"] = ""
        pmf_rows["location"] = loc
        pmf_rows["output_type"] = "pmf"
        pmf_rows["output_type_id"] = pmf_rows["peak_date"].dt.strftime("%Y-%m-%d")
        pmf_rows = pmf_rows.drop(columns=["peak_date"])
        out_rows.append(pmf_rows)

        # ----------------- Peak incidence quantiles (using s_lat only) -----------------
        if obs_g.empty:
            # no observed history: winner value = future latent max per latent path
            peak_idx_lat = g_lat.groupby("sample_id")["value"].idxmax()
            peaks_lat = g_lat.loc[peak_idx_lat, ["sample_id", "value"]].copy()
            winner_peak_vals = peaks_lat["value"].to_numpy(dtype=float)
        else:
            # observed current peak (first max → earliest date on ties)
            obs_sorted = obs_g.sort_values(["value", "date"], ascending=[False, True])
            y_star = float(obs_sorted.iloc[0]["value"])

            # future latent max per latent path
            peak_idx_lat = g_lat.groupby("sample_id")["value"].idxmax()
            fut_lat = g_lat.loc[peak_idx_lat, ["sample_id", "value"]].copy()
            fut_lat.rename(columns={"value": "future_max_lat"}, inplace=True)

            # strict dethrone check on latent values
            dethroned_lat = (fut_lat["future_max_lat"] > y_star)

            # winner peak VALUE is latent: either observed peak (y_star) or latent future max
            winner_peak_vals = np.where(dethroned_lat.to_numpy(),
                                        fut_lat["future_max_lat"].to_numpy(dtype=float),
                                        y_star)

        if winner_peak_vals.size > 0:
            qvals = np.quantile(winner_peak_vals, quantiles)
            qdf = pd.DataFrame({
                "reference_date": ref_date,
                "target": "peak inc flu hosp",
                "horizon": "",
                "target_end_date": "",
                "location": loc,
                "output_type": "quantile",
                "output_type_id": np.array(quantiles, dtype=float),
                "value": qvals.astype(float)
            })
            out_rows.append(qdf)

    # unify and sort without dtype conflicts
    out = pd.concat(out_rows, ignore_index=True)

    pmf = out[out["target"] == "peak week inc flu hosp"].copy()
    qnt = out[out["target"] == "peak inc flu hosp"].copy()

    # PMF: output_type_id are strings 'YYYY-MM-DD'
    pmf = pmf.sort_values(
        ["reference_date", "location", "target", "output_type", "output_type_id"]
    )

    # Quantiles: numeric order
    qnt["output_type_id"] = pd.to_numeric(qnt["output_type_id"], errors="coerce")
    qnt = qnt.sort_values(
        ["reference_date", "location", "target", "output_type", "output_type_id"]
    )

    out = pd.concat([pmf, qnt], ignore_index=True)

    # enforce PMF sums == 1 per (reference_date, location)
    is_pmf = out["target"].eq("peak week inc flu hosp")
    if is_pmf.any():
        sums = out.loc[is_pmf].groupby(["reference_date","location"])["value"].transform("sum")
        out.loc[is_pmf, "value"] = np.where(sums > 0, out.loc[is_pmf, "value"] / sums, out.loc[is_pmf, "value"])

    # final column order
    out = out[[
        "reference_date","target","horizon","target_end_date",
        "location","output_type","output_type_id","value"
    ]]
    return out


In [ ]:
def plot_peak_week_pmf(df_peak: pd.DataFrame, locations=None, loc2abbr=None):
    """
    Plot the peak-week probability mass function (PMF) for given locations.
    If locations=None, plots all locations found in df_peak.

    Expects df_peak columns:
      ['reference_date','target','horizon','target_end_date',
       'location','output_type','output_type_id','value']
    """
    # normalize dtypes
    d = df_peak.copy()
    d = d[d.target=="peak week inc flu hosp"]
    d["location"] = d["location"].astype(str).str.zfill(2)
    d["target_end_date"] = pd.to_datetime(d["target_end_date"])
    d["output_type_id"] = pd.to_datetime(d["output_type_id"])
    # d["horizon"] = pd.to_numeric(d["horizon"], errors="coerce").astype(int)

    # choose locations
    if locations is None:
        locs = sorted(d["location"].unique())
    else:
        locs = [str(x).zfill(2) for x in (locations if isinstance(locations, (list, tuple, set)) else [locations])]

    # plot each location in its own figure
    for loc in locs:
        g = d[d["location"] == loc].copy()
        if g.empty:
            continue

        # ensure sums to 1 (just for safety in the plot)
        sums = g.groupby(["reference_date","location"])["value"].transform("sum")
        g["value"] = g["value"] / sums

        # order x-axis
        # if use_dates:
        g = g.sort_values(["reference_date","output_type_id"])
        x = g["output_type_id"]
        x_labels = g["output_type_id"].dt.strftime("%Y-%m-%d")
        # else:
            # g = g.sort_values(["reference_date","output_type_id"])
            # x = g["horizon"]
            # x_labels = g["horizon"].astype(str)

        y = g["value"].to_numpy()

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.bar(range(len(x)), y, width=0.9, align="center")
        ax.set_xticks(range(len(x)))
        ax.set_xticklabels(x_labels, rotation=45, ha="right")

        locname = loc2abbr[loc] if not None else loc
        ref_dates = g["reference_date"].dt.strftime("%Y-%m-%d").unique()
        title_ref = ref_dates[0] if len(ref_dates) == 1 else f"{ref_dates.min()} … {ref_dates.max()}"
        ax.set_title(f"Peak-week PMF — location {locname} (ref={title_ref})")
        ax.set_xlabel("date")
        ax.set_ylabel("Probability")
        ax.set_ylim(0, max(0.01, y.max()*1.1))
        ax.grid(True, axis="y", alpha=0.3)
        plt.tight_layout()
        plt.show()


In [ ]:
def plot_peak_incidence_quantiles(
    df_peak: pd.DataFrame,
    locations=None,
    ref_date=None,
    show_points=True,
    loc2abbr=None
):
    """
    Plot peak-incidence quantiles from the dataframe returned by
    season_peak_week_pmf_and_peak_quantiles(...).

    Parameters
    ----------
    df_peak : DataFrame
        Must contain rows with:
          target == 'peak inc flu hosp', output_type == 'quantile',
          columns: ['reference_date','location','output_type_id','value', ...]
        where output_type_id holds the quantile level (float).
    locations : list[str|int] | None
        Locations to plot; None -> all locations found in df_all.
    ref_date : str|pd.Timestamp|None
        Reference date to plot. If None, uses the latest reference_date per location.
    show_points : bool
        If True, dots are shown at all reported quantiles.

    Returns
    -------
    None (shows matplotlib figures)
    """
    d = df_peak.copy()
    d = d[d.target=="peak inc flu hosp"]

    # filter to quantile target
    d = d[(d["target"] == "peak inc flu hosp") & (d["output_type"] == "quantile")].copy()
    if d.empty:
        print("No quantile rows found for target='peak inc flu hosp'.")
        return

    # normalize dtypes
    d["reference_date"] = pd.to_datetime(d["reference_date"])
    d["location"] = d["location"].astype(str).str.zfill(2)
    # quantile levels as float
    d["q"] = pd.to_numeric(d["output_type_id"], errors="coerce")
    d = d.dropna(subset=["q"])

    # choose locations
    if locations is None:
        locs = sorted(d["location"].unique())
    else:
        locs = [str(x).zfill(2) for x in (locations if isinstance(locations, (list, tuple, set)) else [locations])]

    # optional fixed ref_date
    fixed_ref = pd.to_datetime(ref_date) if ref_date is not None else None

    def _get_q(series, q):
        """Return value at quantile q if present, else None."""
        s = series.set_index("q")["value"]
        # find closest available quantile (exact match preferred)
        if q in s.index:
            return float(s.loc[q])
        # try near match (handles rounding like 0.1 vs 0.10)
        idx = s.index.to_numpy()
        j = np.argmin(np.abs(idx - q))
        return float(s.iloc[j]) if (len(idx) and np.isclose(idx[j], q, atol=1e-6)) else None

    for loc in locs:
        dloc = d[d["location"] == loc]
        if dloc.empty:
            continue

        # pick ref_date
        if fixed_ref is None:
            rdate = dloc["reference_date"].max()
        else:
            rdate = fixed_ref
        g = dloc[dloc["reference_date"] == rdate].copy()
        if g.empty:
            # nothing for this ref_date/location
            continue

        g = g.sort_values("q")
        qs = g["q"].to_numpy()
        vals = g["value"].to_numpy()

        # pull common bands if available
        q10 = _get_q(g[["q","value"]], 0.10)
        q25 = _get_q(g[["q","value"]], 0.25)
        q50 = _get_q(g[["q","value"]], 0.50)
        q75 = _get_q(g[["q","value"]], 0.75)
        q90 = _get_q(g[["q","value"]], 0.90)

        fig, ax = plt.subplots(figsize=(8, 4.2))

        # fan chart: fill 10–90 and 25–75 using stair-step along quantile axis
        # (We just draw straight fills using the min/max available band levels.)
        if (q10 is not None) and (q90 is not None):
            ax.fill_between([0.10, 0.90], [q10, q10], [q90, q90], alpha=0.15, step="mid", label="10–90%")
        if (q25 is not None) and (q75 is not None):
            ax.fill_between([0.25, 0.75], [q25, q25], [q75, q75], alpha=0.25, step="mid", label="25–75%")

        # full quantile curve
        ax.plot(qs, vals, linewidth=1.5, label="Quantile curve")

        # median line
        if q50 is not None:
            ax.axhline(q50, linestyle="--", linewidth=1.5, label="Median (q=0.5)")

        # optional dots at available quantiles
        if show_points:
            ax.scatter(qs, vals, s=15, zorder=3)

        # axes, title, legend
        locname = loc2abbr[loc] if not None else loc
        ax.set_xlim(0, 1)
        ax.set_xlabel("Quantile level (q)")
        ax.set_ylabel("Peak incidence")
        title_ref = rdate.strftime("%Y-%m-%d")
        ax.set_title(f"Peak-incidence quantiles — location {locname}  (ref={title_ref})")
        ax.grid(True, axis="both", alpha=0.3)
        ax.legend(loc="best", frameon=False)
        plt.tight_layout()
        plt.show()


In [ ]:
# df_peak_forecasts = season_peak_week_pmf_and_peak_quantiles(df_pred_samples, df_hosp_ex_long, quantiles, augment=False)
df_peak_forecasts_aug = season_peak_week_pmf_and_peak_quantiles(df_pred_samples, df_hosp_ex_long, quantiles, augment=True)
df_peak_forecasts_aug2 = season_peak_week_pmf_and_peak_quantiles(df_pred_samples2, df_hosp_ex_long, quantiles, augment=True)

In [ ]:
# plot_peak_week_pmf(df_peak_forecasts_aug, loc2abbr=loc2abbr)
# plot_peak_incidence_quantiles(df_peak_forecasts_aug, loc2abbr=loc2abbr)

loc = 'US' #'06' #
plot_peak_week_pmf(df_peak_forecasts_aug, locations=[loc],loc2abbr=loc2abbr)
plot_peak_week_pmf(df_peak_forecasts_aug2, locations=[loc],loc2abbr=loc2abbr)
# plot_peak_week_pmf(df_peak_forecasts_aug,loc2abbr=loc2abbr)
# plot_peak_week_pmf(df_peak_forecasts_aug2,loc2abbr=loc2abbr)

plot_peak_incidence_quantiles(df_peak_forecasts_aug, locations=[loc], loc2abbr=loc2abbr)
plot_peak_incidence_quantiles(df_peak_forecasts_aug2, locations=[loc], loc2abbr=loc2abbr)

In [ ]:
weights = np.array([1.0, 0.0]) #np.array([0.9, 0.1])
weights = weights/np.sum(weights)
df_peak_forecasts_combined = df_peak_forecasts_aug.copy()
df_peak_forecasts_combined['value'] = weights[0]*df_peak_forecasts_aug['value']+weights[1]*df_peak_forecasts_aug2['value']

loc = 'US' #
# plot_peak_week_pmf(df_peak_forecasts_combined, locations=[loc],loc2abbr=loc2abbr)
# plot_peak_incidence_quantiles(df_peak_forecasts_combined, locations=[loc], loc2abbr=loc2abbr)
plot_peak_week_pmf(df_peak_forecasts_combined,loc2abbr=loc2abbr)
plot_peak_incidence_quantiles(df_peak_forecasts_combined, loc2abbr=loc2abbr)

In [ ]:
def check_peak_week_pmf_sums(df, tol=1e-8):
    mask = (
        (df["target"] == "peak week inc flu hosp") &
        (df["output_type"] == "pmf")
    )
    sub = df[mask].copy()
    # Sum values by (reference_date, location)
    sums = (
        sub
        .groupby(["reference_date", "location"], as_index=False)["value"]
        .sum()
        .rename(columns={"value": "sum_value"})
    )
    # Check which groups deviate from 1
    bad = sums[np.abs(sums["sum_value"] - 1.0) > tol]
    if bad.empty:
        print("OK ✅ All peak-week PMFs sum to 1 within tolerance.")
    else:
        print("❌ Some groups do NOT sum to 1:")
        print(bad.head())

def renormalize_peak_week_pmf(df):
    mask = (
        (df["target"] == "peak week inc flu hosp") &
        (df["output_type"] == "pmf")
    )
    df.loc[mask, "value"] = (
        df.loc[mask]
        .groupby(["reference_date", "location"])["value"]
        .transform(lambda x: x / x.sum() if x.sum() != 0 else x)
    )
    return df

# check_peak_week_pmf_sums(df_peak_forecasts_combined)
df_peak_forecasts_combined = renormalize_peak_week_pmf(df_peak_forecasts_combined)
check_peak_week_pmf_sums(df_peak_forecasts_combined)

In [ ]:
q_allowed_str = [
    '0.01', '0.025', '0.05', '0.1', '0.15', '0.2', '0.25', '0.3', '0.35',
    '0.4', '0.45', '0.5', '0.55', '0.6', '0.65', '0.7', '0.75', '0.8',
    '0.85', '0.9', '0.95', '0.975', '0.99'
]
q_allowed = np.array(list(map(float, q_allowed_str)))

def snap_to_q_allowed(q):
    v = float(q)
    idx = np.argmin(np.abs(q_allowed - v))
    return q_allowed_str[idx]

mask = df_peak_forecasts_combined['target'] == 'peak inc flu hosp'
df_peak_forecasts_combined.loc[mask, 'output_type_id'] = df_peak_forecasts_combined.loc[mask, 'output_type_id'].map(snap_to_q_allowed)

sorted(df_peak_forecasts_combined.loc[df_peak_forecasts_combined['target'] == 'peak inc flu hosp', 'output_type_id'].unique())

In [ ]:
df_peak_forecasts_combined['location'] = df_peak_forecasts_combined['location'].astype(str).str.zfill(2)
df_peak_forecasts_combined.to_csv(f"{results_dir}peak_forecasts/{format(ref_date,'%Y-%m-%d')}.csv",index=False)